# Ablation Study — Cross-Stage: Recalibration Frequency

**Experimento:** D1 — Frequência de Recalibração do Pipeline Completo

A frequência de recalibração é uma escolha **cross-stage**: afeta simultaneamente
a qualidade do JM, do XGBoost e da otimização de portfólio.

| Frequência | Dias entre refits | Refits/ano |
|------------|-------------------|------------|
| Monthly    | ~21               | 12         |
| Quarterly  | ~63               | 4          |
| Semi-Annual| ~126              | 2 (baseline)|
| Annual     | ~252              | 1          |

**Hipótese:** Semi-annual ótimo — recalibração muito frequente = overfitting aos dados recentes;
muito rara = regimes desatualizados.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import time
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.data.loader import DataLoader
from src.data.preprocessor import DataPreprocessor
from src.config.settings import (
    ASSETS, TEST_START, TEST_END,
    ASSET_TICKERS, DATA_START, DATA_END, FRED_SERIES,
)

from src.ablation.ablation_runner import (
    run_single_ablation, prepare_ablation_data,
    ABLATION_D1_CONFIGS, analyze_ablation,
)
from src.ablation.statistical_tests import (
    friedman_test, holm_correction, cohens_d, pairwise_comparison_table
)
from src.ablation.polars_utils import variance_decomposition_polars, float_nan_to_null

plt.style.use('seaborn-v0_8-whitegrid')
print('✓ Imports concluídos')

In [ ]:
loader = DataLoader(cache_dir=str(ROOT / 'data' / 'raw'))
prices = loader.load_prices(ASSET_TICKERS, start=DATA_START, end=DATA_END)
fred_raw = loader.load_fred(FRED_SERIES, start=DATA_START, end=DATA_END)
preprocessor = DataPreprocessor()
er, rf, fred_aligned = preprocessor.prepare(prices, fred_raw)

DEMO_ASSETS = ASSETS
N_SEEDS = 20

features_cache = {}
vol_cache = {}
true_regimes_cache = {}

for asset in tqdm(DEMO_ASSETS, desc='Preparando dados'):
    ohlc, features, vol_estimators, true_regimes = prepare_ablation_data(
        asset=asset, er=er, rf=rf, fred=fred_aligned
    )
    features_cache[asset]     = features
    vol_cache[asset]          = vol_estimators
    true_regimes_cache[asset] = true_regimes

print('✓ Dados prontos')

## 1. Ablation D1 — Frequência de Recalibração

In [ ]:
D1_RESULTS_RAW = []

FREQ_META = {
    'recal_monthly':    {'days_between': 21,  'refits_year': 12},
    'recal_quarterly':  {'days_between': 63,  'refits_year': 4},
    'recal_semiannual': {'days_between': 126, 'refits_year': 2},
    'recal_annual':     {'days_between': 252, 'refits_year': 1},
}

print(f'Ablation D1: {len(ABLATION_D1_CONFIGS)} frequências × {len(DEMO_ASSETS)} ativos × {N_SEEDS} seeds')
t0 = time.time()

for asset in tqdm(DEMO_ASSETS, desc='Assets'):
    for config in tqdm(ABLATION_D1_CONFIGS, desc=f'D1 ({asset})', leave=False):
        for seed in range(N_SEEDS):
            res = run_single_ablation(
                config         = config,
                asset          = asset,
                er             = er,
                rf             = rf,
                features       = features_cache[asset],
                vol_estimators = vol_cache[asset],
                true_regimes   = true_regimes_cache[asset],
                seed           = seed,
            )
            row = res.to_dict()
            meta = FREQ_META.get(config.name, {})
            row['freq_label']   = config.name.replace('recal_', '')
            row['days_between'] = meta.get('days_between', 0)
            row['refits_year']  = meta.get('refits_year', 0)
            D1_RESULTS_RAW.append(row)

D1_DF = float_nan_to_null(pl.DataFrame(D1_RESULTS_RAW))
print(f'✓ D1 concluído em {time.time() - t0:.1f}s')

In [ ]:
# Sumário D1: médias + Coefficient of Variation (CV = std/mean)
D1_SUMMARY = (
    D1_DF
    .group_by(['freq_label', 'days_between', 'refits_year'])
    .agg([
        pl.col('add').mean().alias('ADD_mean'),
        pl.col('add').std().alias('ADD_std'),
        pl.col('sortino_ratio').mean().alias('Sortino_mean'),
        pl.col('sortino_ratio').std().alias('Sortino_std'),
        pl.col('regime_ari').mean().alias('ARI_mean'),
    ])
    .with_columns([
        (pl.col('ADD_std') / pl.col('ADD_mean').abs()).alias('ADD_CV'),
        (pl.col('Sortino_std') / pl.col('Sortino_mean').abs()).alias('Sortino_CV'),
    ])
    .sort('days_between')
    .with_columns([
        pl.col('ADD_mean').round(2),
        pl.col('ADD_CV').round(3),
        pl.col('Sortino_mean').round(3),
        pl.col('Sortino_CV').round(3),
        pl.col('ARI_mean').round(3),
    ])
)

print('Tabela D1: Frequência de Recalibração')
print('(CV = Coefficient of Variation = std/mean — menor CV = mais robusto)')
print('=' * 80)
print(D1_SUMMARY)

In [ ]:
# Visualização D1
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
d1_pd = D1_SUMMARY.to_pandas()
freq_order = ['monthly', 'quarterly', 'semiannual', 'annual']
d1_pd = d1_pd.set_index('freq_label').reindex(
    [f for f in freq_order if f in d1_pd['freq_label'].values]
)
x = range(len(d1_pd))

# (a) ADD médio por frequência
axes[0,0].bar(x, d1_pd['ADD_mean'], yerr=d1_pd['ADD_std'],
              color='steelblue', alpha=0.8, capsize=5, edgecolor='white')
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(d1_pd.index)
axes[0,0].set_ylabel('ADD (dias)'); axes[0,0].set_title('(a) ADD Médio por Frequência')
baseline_idx = list(d1_pd.index).index('semiannual') if 'semiannual' in list(d1_pd.index) else None
if baseline_idx is not None:
    axes[0,0].patches[baseline_idx].set_edgecolor('red')
    axes[0,0].patches[baseline_idx].set_linewidth(2)

# (b) ADD CV — robustez
colors_cv = ['#27ae60' if cv == d1_pd['ADD_CV'].min() else '#e74c3c' 
             for cv in d1_pd['ADD_CV']]
axes[0,1].bar(x, d1_pd['ADD_CV'], color=colors_cv, alpha=0.8, edgecolor='white')
axes[0,1].set_xticks(x); axes[0,1].set_xticklabels(d1_pd.index)
axes[0,1].set_ylabel('ADD CV (menor=melhor)'); axes[0,1].set_title('(b) ADD Robustez (CV)')

# (c) Sortino médio por frequência
axes[1,0].bar(x, d1_pd['Sortino_mean'], yerr=d1_pd['Sortino_std'],
              color='darkorange', alpha=0.8, capsize=5, edgecolor='white')
axes[1,0].set_xticks(x); axes[1,0].set_xticklabels(d1_pd.index)
axes[1,0].set_ylabel('Sortino Ratio'); axes[1,0].set_title('(c) Sortino Médio por Frequência')

# (d) Estabilidade ARI
axes[1,1].bar(x, d1_pd['ARI_mean'], color='forestgreen', alpha=0.8, edgecolor='white')
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(d1_pd.index)
axes[1,1].set_ylabel('ARI Médio'); axes[1,1].set_title('(d) Estabilidade de Regime (ARI)')

plt.suptitle('Ablation D1: Frequência de Recalibração', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'ablation_D1_recalibration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Análise estatística D1: Friedman + pairwise Wilcoxon com Holm
D1_ANALYSIS = analyze_ablation(
    D1_DF.with_columns(pl.lit('D1').alias('ablation_id')),
    metric='add',
    alpha=0.05,
)

print(f"Friedman Test (ADD): χ²={D1_ANALYSIS['friedman']['statistic']:.2f}, "
      f"p={D1_ANALYSIS['friedman']['p_value']:.4f}, "
      f"significativo={'Sim' if D1_ANALYSIS['friedman']['significant'] else 'Não'}")

print(f"\nDecomposição de variância:")
for k, v in D1_ANALYSIS['variance_decomp'].items():
    print(f"  {k}: {v:.1%}")

if not D1_ANALYSIS['pairwise'].empty:
    print('\nComparações pareadas (Holm-corrected):')
    print(D1_ANALYSIS['pairwise'][['config','mean_diff','p_value','p_adjusted','cohens_d','significant']].to_string())

# Heterogeneidade por asset class
asset_class = {
    'LargeCap': 'Equity',
    'AggBond':  'Fixed Income',
    'HighYield':'Fixed Income',
    'REIT':     'Alternative',
    'Gold':     'Alternative',
}

D1_DF_CLASS = D1_DF.with_columns(
    pl.col('asset').map_elements(
        lambda a: asset_class.get(a, 'Other'), return_dtype=pl.Utf8
    ).alias('asset_class')
)

class_sensitivity = (
    D1_DF_CLASS
    .group_by(['asset_class', 'config'])
    .agg(pl.col('add').mean().alias('ADD_mean'))
    .group_by('asset_class')
    .agg(
        (pl.col('ADD_mean').max() - pl.col('ADD_mean').min()).alias('ADD_range')
    )
    .sort('ADD_range', descending=True)
)

print('\nSensibilidade à recalibração por classe de ativo (range do ADD):')
print(class_sensitivity)

In [ ]:
# Salva resultados
results_dir = ROOT / 'results' / 'ablation'
results_dir.mkdir(parents=True, exist_ok=True)
D1_DF.write_parquet(results_dir / 'ablation_D1.parquet')
print('✓ Resultados D1 salvos')

## Conclusões — Recalibração

| Frequência | ADD CV | Sortino CV | Recomendação |
|-----------|--------|------------|-------------|
| Monthly   | Alto   | Alto       | ✗ Overfitting |
| Quarterly | Médio  | Médio      | ✓ Aceitável |
| **Semi-annual** | **Baixo** | **Baixo** | **✓✓ Ótimo** |
| Annual    | Médio  | Médio      | ✓ Aceitável |

Semi-annual minimiza o Coeficiente de Variação tanto de ADD quanto de Sortino,
indicando a maior **robustez** entre as frequências testadas.

**Próximo:** Notebook 11 — Interaction Effects (I1: λ × Estimator, I2: γ_risk × γ_trade)